# Apply current DonSheet formatting to an existing DonSheet

Use this notebook when a filled DonSheet needs its visual formatting restored. It applies the current v20261018 formatting conventions without changing worksheet content.

The notebook does **not** assign cell values, add or replace formulas, rebuild dropdown lists or validations, alter worksheet protection, rename sheets, or add/delete rows or columns. It verifies after saving that every non-empty cell has the same value, formula status, and formula text as the input workbook.

In [ ]:
from pathlib import Path

# Change this to the filled DonSheet that should receive the formatting.
INPUT_FILE = Path('/Users/oliverglanz/Desktop/DonSheet_202821-Spring_v1.0.xlsx')
OUTPUT_FILE = INPUT_FILE.with_name(f'{INPUT_FILE.stem}_formatted.xlsx')

# Match the formula/template range used by the current DonSheet.
MAX_FORMAT_ROWS = 300

if not INPUT_FILE.is_file():
    raise FileNotFoundError(f'Input DonSheet not found: {INPUT_FILE}')

print(f'Input:  {INPUT_FILE}')
print(f'Output: {OUTPUT_FILE}')


In [ ]:
from hashlib import sha256
import re
import zipfile

from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import column_index_from_string, get_column_letter

SKIP_SHEETS = {'DropDownMenu', 'InstructorMap'}
DATE_COLUMN_HEADERS = {
    'Semester Start Date {Soaterm Start Date}',
    'Pre-work Start Date {not in Banner}',
    'Pre-work End Date {not in Banner}',
    'Course Start Date {Meet Start Date}',
    'Course End Date {Meet End Date}',
    # Legacy headers remain supported for previously created DonSheets.
    'Intensive Period Start Date {Meet Start Date}',
    'Intensive Period End Date {Meet End Date}',
    'Post-work Start Date {not in Banner}',
    'Post-work End Date {not in Banner}',
    'Semester End Date {Soaterm End Date}',
}
DATE_NUMBER_FORMAT = 'd-mmm-yy'  # Excel displays this as 14-Mar-26.
TEXT_COLUMN_HEADERS = {
    'Notes {not in Banner}', 'CRN sorted {not in Banner}', 'CRN',
    'Attribute {not in Banner}', 'Subject', 'Course Number {Crse Num}',
    'Course Section {Seq Crse Num}', 'Program {not in Banner}', 'Catalog Title',
    'Section Title', 'Crosslist Details {not in Banner}', 'X Lst',
    'Campus {Ssasect Campus}', 'Campus Restriction {Camp Restr CC}',
    'Schedule Type {not in Banner}', 'Instruction Method {Inst Method}',
    'Level {Level SC}', 'Part-Of-Term', 'Credit Catalog {Cat Crs}',
    'Meeting Type', 'Course Beginning Time {Meet Beg Time}',
    'Course Ending Time {Meet End Time}', 'SUN', 'MON', 'TUE', 'WED',
    'THU', 'FRI', 'SAT', 'Fiscal Year {not in Banner}',
    'Semester {not in Banner}', 'Room {Meet Room}', 'Building {Meet Bldg}',
    'Instructor Name {Instr Name}', 'Instructor Email {Instr Email}',
    'Instructor ID {Instr ID}', 'Remote Employee {not in Banner}',
    'load/contract {not in Banner}', 'Reason for Contract {not in Banner}',
    'dept budget {not in Banner}', 'account to be charged {not in Banner}',
    'SEM Department {Scacrse Dept}', 'dean email', 'VP finance email',
    'HR email', 'Fee Amount {Fees Amt}', 'Fee Code {Detl Code}', 'Fee Type',
    'Fee Level', 'Fee Ind', 'Fee Term', 'Activity Date', 'Status',
    'Bill Hrs', 'Sect Sch Type', 'Prerequisites {Preq Areas}', 'Link Ind',
    'Integration Code', 'Link Conn', 'Crs Attr', 'Comments', 'Major Restr',
    'Scacrse College', 'Enrolled', 'Waitlist Capacity', 'Primary Ind',
    'Override', '1st Date Reg Opens', 'Last Date Reg Opens',
    'OL Range From Date', 'OL Range To Date', 'OL Numb Units',
    'OL Duration Code', 'Grade Mode', 'Gradable Ind', 'Tuit Waiver',
    'Course Override {Meet Override}', 'Lvl Res Ind', 'Cmp Res Ind',
    'Rate Code', 'Cohort Code', 'St Attr Code', 'Coll SC', 'DegC SC',
    'Prog SC', 'FOS Type', 'Special Approval {Special Apvl}', 'Term Code',
    'Camp SC',
}
BLUE_HEADERS = {
    'pre-work # of weeks {not in Banner}', 'pre-work hours/week {not in Banner}',
    'pre-work hours/period {not in Banner}', 'intensive # of weeks {not in Banner}',
    'intensive hours/week {not in Banner}', 'intensive hours/period {not in Banner}',
    'post-work # of weeks {not in Banner}', 'post-work hours/week {not in Banner}',
    'post-work hours/period {not in Banner}', 'sum of weeks (for check) {not in Banner}',
    'total contract hours {not in Banner}', 'Year {not in Banner}',
    'Semester {not in Banner}', 'Remote Employee {not in Banner}',
    '% Responsibility, load/contract {not in Banner}',
    'Reason for Contract {not in Banner}', 'costs per credit {not in Banner}',
    'total costs {not in Banner}', 'dept budget {not in Banner}',
    'account to be charged {not in Banner}', 'SEM Department {Scacrse Dept}',
}
PURPLE_HEADERS = {
    'pre-work # of weeks {not in Banner}', 'pre-work hours/period {not in Banner}',
    'intensive # of weeks {not in Banner}', 'intensive hours/period {not in Banner}',
    'post-work # of weeks {not in Banner}', 'post-work hours/week {not in Banner}',
    'post-work hours/period {not in Banner}', 'sum of weeks (for check) {not in Banner}',
    'Fiscal Year {not in Banner}', 'Semester {not in Banner}',
    'total costs {not in Banner}',
}
GREY_HEADERS = {'CRN sorted {not in Banner}', 'CRN', 'X Lst'}
FIXED_WIDTHS = {
    'Notes {not in Banner}': 40, 'Program {not in Banner}': 12,
    'Catalog Title': 25, 'Section Title': 25, 'Schedule Type {not in Banner}': 15,
    'Instruction Method {Inst Method}': 10, 'Building {Meet Bldg}': 30,
    'Instructor Name {Instr Name}': 20, 'Instructor Email {Instr Email}': 20,
    'Instructor ID {Instr ID}': 10, 'load/contract {not in Banner}': 15,
    'Reason for Contract {not in Banner}': 15,
    'account to be charged {not in Banner}': 25,
}
GROUP_DEFS = [('R', 'AB'), ('AD', 'AI'), ('AK', 'AU'), ('AY', 'BJ'),
              ('BK', 'BV'), ('BX', 'CE'), ('CG', 'DQ')]

THIN_GREY_BORDER = Border(
    left=Side(style='thin', color='D3D3D3'),
    right=Side(style='thin', color='D3D3D3'),
    top=Side(style='thin', color='D3D3D3'),
    bottom=Side(style='thin', color='D3D3D3'),
)

def content_signature(workbook):
    """Hash non-empty cells without looking at formatting."""
    digest = sha256()
    for worksheet in workbook.worksheets:
        digest.update(worksheet.title.encode())
        for _, cell in sorted(worksheet._cells.items()):
            if cell.value is None:
                continue
            digest.update(f'{cell.coordinate}|{cell.data_type}|{cell.value!r}'.encode())
    return digest.hexdigest()

def capture_worksheet_extensions(workbook_path):
    """Preserve modern Excel worksheet extensions unsupported by openpyxl."""
    extensions = {}
    with zipfile.ZipFile(workbook_path) as archive:
        for part_name in archive.namelist():
            if not re.fullmatch(r'xl/worksheets/sheet\d+\.xml', part_name):
                continue
            xml = archive.read(part_name).decode('utf-8')
            extension = re.search(r'<extLst>.*?</extLst>', xml, flags=re.DOTALL)
            root = re.search(r'<worksheet\b[^>]*>', xml)
            if extension and root:
                namespaces = re.findall(r'\s(xmlns:[^=]+)="([^"]+)"', root.group(0))
                extensions[part_name] = (extension.group(0), namespaces)
    return extensions

def restore_worksheet_extensions(workbook_path, extensions):
    if not extensions:
        return
    temporary_path = workbook_path.with_name(workbook_path.stem + '_extensions_temp.xlsx')
    with zipfile.ZipFile(workbook_path) as source, zipfile.ZipFile(temporary_path, 'w', zipfile.ZIP_DEFLATED) as target:
        for entry in source.infolist():
            data = source.read(entry.filename)
            if entry.filename in extensions:
                xml = data.decode('utf-8')
                extension, namespaces = extensions[entry.filename]
                xml = re.sub(r'<extLst>.*?</extLst>', '', xml, flags=re.DOTALL)
                root = re.search(r'<worksheet\b[^>]*>', xml)
                if root:
                    root_tag = root.group(0)
                    for attribute, namespace in namespaces:
                        if attribute not in root_tag:
                            root_tag = root_tag[:-1] + f' {attribute}="{namespace}">'
                    xml = xml[:root.start()] + root_tag + xml[root.end():]
                data = xml.replace('</worksheet>', extension + '</worksheet>').encode('utf-8')
            target.writestr(entry, data)
    temporary_path.replace(workbook_path)

def used_header_count(worksheet):
    return max(
        (index for index, cell in enumerate(worksheet[1], start=1)
         if cell.value is not None and str(cell.value).strip()),
        default=1,
    )

def set_font(cell, *, size, bold=None, italic=None):
    old = cell.font or Font()
    cell.font = Font(
        name='Arial', size=size,
        bold=old.bold if bold is None else bold,
        italic=old.italic if italic is None else italic,
        underline=old.underline, color=old.color,
    )

def apply_formatting(worksheet):
    headers = [cell.value for cell in worksheet[1]]
    last_column = used_header_count(worksheet)
    headers = headers[:last_column]
    last_row = MAX_FORMAT_ROWS

    notes_index = headers.index('Notes {not in Banner}') + 1 if 'Notes {not in Banner}' in headers else 0
    worksheet.freeze_panes = f'{get_column_letter(notes_index + 1)}2' if notes_index else 'A2'
    worksheet.auto_filter.ref = f'A1:{get_column_letter(last_column)}{last_row}'

    italic_header_fields = {
        'Semester Start Date {Soaterm Start Date}',
        'Semester End Date {Soaterm End Date}',
    }
    default_fill = PatternFill(start_color='D3D3D3', end_color='D3D3D3', fill_type='solid')
    for cell in worksheet[1]:
        if cell.value is None or not str(cell.value).strip():
            continue
        cell.font = Font(
            name='Arial', size=10, bold=cell.value not in italic_header_fields,
            italic=cell.value in italic_header_fields,
            color='FF0000' if cell.value == 'Notes {not in Banner}' else '000000',
        )
        cell.fill = default_fill
        cell.alignment = Alignment(horizontal='center', vertical='bottom', wrap_text=True, text_rotation=90)
    worksheet.row_dimensions[1].height = 150

    if 'Don {not in Banner}' in headers and 'SEM Department {Scacrse Dept}' in headers:
        light_blue = PatternFill(start_color='DDEBF7', end_color='DDEBF7', fill_type='solid')
        for column in range(headers.index('Don {not in Banner}') + 1, headers.index('SEM Department {Scacrse Dept}') + 2):
            worksheet.cell(1, column).fill = light_blue
    blue_fill = PatternFill(start_color='00B0F0', end_color='00B0F0', fill_type='solid')
    purple_fill = PatternFill(start_color='7A81FF', end_color='7A81FF', fill_type='solid')
    grey_fill = PatternFill(start_color='D3D3D3', end_color='D3D3D3', fill_type='solid')
    for column, header in enumerate(headers, start=1):
        if header in BLUE_HEADERS:
            worksheet.cell(1, column).fill = blue_fill
        if header in PURPLE_HEADERS:
            worksheet.cell(1, column).fill = purple_fill
        if header in GREY_HEADERS:
            worksheet.cell(1, column).fill = grey_fill

    for row in worksheet.iter_rows(min_row=2, max_row=last_row, min_col=1, max_col=last_column):
        for cell in row:
            cell.border = THIN_GREY_BORDER
            set_font(cell, size=12 if cell.column <= 7 else 10)

    for column, header in enumerate(headers, start=1):
        if header in italic_header_fields:
            for row in range(2, last_row + 1):
                set_font(worksheet.cell(row, column), size=10, bold=False, italic=False)
        if header in TEXT_COLUMN_HEADERS:
            for row in range(2, last_row + 1):
                worksheet.cell(row, column).number_format = '@'
        if header in DATE_COLUMN_HEADERS:
            for row in range(2, last_row + 1):
                worksheet.cell(row, column).number_format = DATE_NUMBER_FORMAT

    for column in range(1, last_column + 1):
        max_length = max(
            (len(str(worksheet.cell(row, column).value))
             for row in range(2, min(worksheet.max_row, last_row) + 1)
             if worksheet.cell(row, column).value is not None),
            default=0,
        )
        worksheet.column_dimensions[get_column_letter(column)].width = max(4, min(max_length + 2, 60))
    for header, width in FIXED_WIDTHS.items():
        if header in headers:
            worksheet.column_dimensions[get_column_letter(headers.index(header) + 1)].width = width

    for dimension in worksheet.column_dimensions.values():
        dimension.outline_level = 0
        dimension.hidden = False
        dimension.collapsed = False
    worksheet.sheet_view.showOutlineSymbols = True
    worksheet.sheet_properties.outlinePr.summaryRight = False
    worksheet.sheet_properties.outlinePr.summaryBelow = True
    worksheet.sheet_properties.outlinePr.applyStyles = True
    for start_column, end_column in GROUP_DEFS:
        start = column_index_from_string(start_column)
        end = min(column_index_from_string(end_column), last_column)
        for column in range(start + 1, end + 1):
            worksheet.column_dimensions[get_column_letter(column)].outline_level = 1

worksheet_extensions = capture_worksheet_extensions(INPUT_FILE)
wb = load_workbook(INPUT_FILE, data_only=False, keep_links=True)
before_signature = content_signature(wb)
formatted_sheets = []
for ws in wb.worksheets:
    if ws.title in SKIP_SHEETS or ws.title.startswith('_'):
        continue
    apply_formatting(ws)
    formatted_sheets.append(ws.title)

wb.save(OUTPUT_FILE)
wb.close()
restore_worksheet_extensions(OUTPUT_FILE, worksheet_extensions)

verification_wb = load_workbook(OUTPUT_FILE, data_only=False, keep_links=True)
after_signature = content_signature(verification_wb)
verification_wb.close()
if before_signature != after_signature:
    raise RuntimeError('Content verification failed: the output workbook differs from the input workbook.')

print(f'Applied formatting to {len(formatted_sheets)} worksheet(s).')
print(f'Content verified unchanged. Saved: {OUTPUT_FILE}')
